# 20 · Query & federation — Trino cross-catalog queries

**Trino is the mesh's federated query engine.** It connects to many storage systems at
once and lets **one SQL statement span all of them** — you query the data *where it
lives*, join across engines that were never designed to talk to each other, and no ETL
job has to copy anything first.

That is the whole thesis of a federated query engine:

> **Query where the data lives. Join across engines. Move no bytes you don't have to.**

The lakehouse (notebooks `10`/`11`) is where curated data is *versioned and stored*.
This notebook is one layer up: it's where that data — plus the platform's live
operational databases — is *queried together*. Where lakeFS and Nessie give you `branch`
/ `commit` / `merge`, Trino gives you `SELECT ... JOIN ...` across catalogs.

### The catalogs wired into this Trino

A Trino **catalog** is a named connection to one backing system. This coordinator has
exactly three:

| catalog | connects to | what's in it |
|---------|-------------|--------------|
| **`iceberg`** | the Nessie/Iceberg **lakehouse** on MinIO | analytical tables — dbt marts, dataset mirrors, eval scores (notebook `11`'s world) |
| **`postgresql`** | the **operational** AI eval/operator Postgres | live rows the platform writes as it runs — eval runs, results, models, operator sessions |
| **`system`** | Trino's **own metadata** | catalogs, schemas, running queries, nodes — Trino describing itself |

The interesting pair is the first two: `iceberg` is the *analytical* store (columnar,
versioned, batch-built) and `postgresql` is the *operational* store (row-oriented, live,
transactional). They are completely different engines. The centerpiece of this notebook
is a **single query that joins across both** — the lakehouse's analytical scores against
the operational database's run/model/latency context — with no pipeline in between.

> **Read-only.** Everything here is `SHOW` / `DESCRIBE` / `SELECT` / `EXPLAIN`. Nothing is
> written to any catalog — and `postgresql` is a **live operational database**, so we never
> `INSERT` / `UPDATE` / `CREATE` there. Because we create nothing, there is **no cleanup
> section** (unlike notebooks `10` and `11`, which branch and must tidy up).

## Setup

The `trino` Python client is **not** in the singleuser base image (which ships `polars`,
`s3fs`, `pyarrow`, `duckdb`, `fastavro`), so we install it here. `polars` — used to render
result frames, exactly as in notebooks `10`/`11` — already ships in the image.

In [1]:
%pip install -q trino


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect

Connection is **env-driven**. The committed defaults are the **in-cluster** service URL
(`trino.data-mesh.svc.cluster.local:8080`); a validation run overrides `TRINO_HOST` /
`TRINO_PORT` (e.g. to a `kubectl port-forward`) without editing the notebook — the same
pattern the storage notebooks use.

We open **one connection with no catalog or schema pinned**, so every query is free to
cross catalogs by fully qualifying names as `catalog.schema.table`. This Trino speaks
plain HTTP on port 8080 and takes an empty password.

`q(sql)` is our tiny helper: run a statement and hand the rows back as a **polars**
DataFrame for display (mirroring how `10`/`11` render frames).

In [2]:
import os
import trino
import polars as pl

TRINO_HOST = os.environ.get("TRINO_HOST", "trino.data-mesh.svc.cluster.local")
TRINO_PORT = int(os.environ.get("TRINO_PORT", "8080"))
TRINO_USER = os.environ.get("TRINO_USER", "jupyter")

conn = trino.dbapi.connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user=TRINO_USER,
    http_scheme="http",   # this coordinator is plain HTTP, unmeshed
    # no catalog/schema pinned -> queries fully-qualify names and can cross catalogs
)

# q(sql)      -> run a statement, return the rows as a polars DataFrame
# scalar(sql) -> run a statement, return the single top-left value
def q(sql):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pl.DataFrame(rows, schema=cols, orient="row")

def scalar(sql):
    cur = conn.cursor()
    cur.execute(sql)
    return cur.fetchone()[0]

print("Trino host   :", f"{TRINO_HOST}:{TRINO_PORT}")
print("Trino version:", scalar("SELECT version()"))   # proves the connection
print("as user      :", TRINO_USER)

Trino host   : localhost:8085


Trino version: 468
as user      : jupyter


## Discover — what can I query?

Trino is self-describing. `SHOW CATALOGS` lists the wired connections; `SHOW SCHEMAS FROM
<catalog>` lists the schemas (databases/namespaces) inside one. This is how you explore a
federated engine before writing a line of analytical SQL — you're browsing several
different backing systems through one interface.

In [3]:
q("SHOW CATALOGS")

Catalog
str
"""iceberg"""
"""postgresql"""
"""system"""


The two catalogs we'll join. **`iceberg`** carries the lakehouse schemas (the same
namespaces notebook `11` browsed through Nessie); **`postgresql`** carries the operational
database's schemas (`public` holds the eval/operator tables).

In [4]:
cats = pl.concat([
    q("SHOW SCHEMAS FROM iceberg").with_columns(catalog=pl.lit("iceberg")),
    q("SHOW SCHEMAS FROM postgresql").with_columns(catalog=pl.lit("postgresql")),
])
cats.select(["catalog", "Schema"])

catalog,Schema
str,str
"""iceberg""","""analytics"""
"""iceberg""","""catalog"""
"""iceberg""","""datasets_health"""
"""iceberg""","""datasets_music"""
"""iceberg""","""dbt"""
"""iceberg""","""eval"""
"""iceberg""","""information_schema"""
"""postgresql""","""information_schema"""
"""postgresql""","""pg_catalog"""


### The two tables we're about to join

The federation query below joins **`iceberg.eval.eval_scores`** (the lakehouse table of
per-metric judge scores) to **`postgresql.public.eval_results`** (the operational table of
per-question model answers, with latency and backend). `DESCRIBE` shows Trino resolving
each table's columns from its *own* backing system — Iceberg metadata for one, the live
Postgres catalog for the other.

The join key is `eval_scores.result_id = eval_results.id`; each result can also be traced
up to its run via `eval_results.run_id = eval_runs.id`.

In [5]:
scores_schema = q("DESCRIBE iceberg.eval.eval_scores").with_columns(source=pl.lit("iceberg.eval.eval_scores"))
results_schema = q("DESCRIBE postgresql.public.eval_results").with_columns(source=pl.lit("postgresql.public.eval_results"))
pl.concat([
    scores_schema.select(["source", "Column", "Type"]),
    results_schema.select(["source", "Column", "Type"]),
])

source,Column,Type
str,str,str
"""iceberg.eval.eval_scores""","""id""","""bigint"""
"""iceberg.eval.eval_scores""","""result_id""","""bigint"""
"""iceberg.eval.eval_scores""","""metric""","""varchar"""
"""iceberg.eval.eval_scores""","""judge""","""varchar"""
"""iceberg.eval.eval_scores""","""score""","""double"""
…,…,…
"""postgresql.public.eval_results""","""answer""","""varchar"""
"""postgresql.public.eval_results""","""contexts""","""json"""
"""postgresql.public.eval_results""","""latency_ms""","""integer"""


## Single-catalog baselines — each source alone first

Before crossing catalogs, look at each side on its own so the federation is unambiguous.

**From the lakehouse (`iceberg`)** — a dbt mart, the same kind of Iceberg table notebook
`11` read through Nessie. Artist popularity, top by play count:

In [6]:
q('''
    SELECT artist_name, total_plays, n_listeners
    FROM iceberg.dbt.mart_artist_popularity
    ORDER BY total_plays DESC
    LIMIT 5
''')

artist_name,total_plays,n_listeners
str,i64,i64
"""the beatles""",24535627,60223
"""radiohead""",22155328,63873
"""coldplay""",13448263,51415
"""pink floyd""",12849101,35355
"""metallica""",12309799,35361


**From the operational database (`postgresql`)** — the live eval-run ledger. These rows
are written by the platform as evaluations execute; Trino reads them straight from
Postgres:

In [7]:
q('''
    SELECT status, count(*) AS n_runs, sum(question_count) AS total_questions
    FROM postgresql.public.eval_runs
    GROUP BY status
    ORDER BY n_runs DESC
''')

status,n_runs,total_questions
str,i64,i64
"""scored""",14,240
"""results_ready""",6,110
"""questions_ready""",3,50


## The federation — one query, two engines

This is the centerpiece. **The lakehouse holds the analytical scores; the operational
Postgres holds the live run/model/latency context** — and Trino joins them in a single
statement, with nothing copied between them:

- `iceberg.eval.eval_scores` (lakehouse) contributes `metric` and `score` — the RAG-eval
  judge output, one row per (result, metric).
- `postgresql.public.eval_results` (operational) contributes `model`, `backend`, and
  `latency_ms` — the live context of *which model produced that answer and how slowly*.
- They meet on `eval_scores.result_id = eval_results.id`.

Without federation you'd need a pipeline to land the Postgres rows into the lakehouse (or
vice versa) before you could join them. Here it's one `JOIN`.

In [8]:
fed = q('''
    SELECT r.model,
           r.backend,
           s.metric,
           count(*)                 AS n,
           round(avg(s.score), 3)   AS avg_score,
           round(avg(r.latency_ms), 0) AS avg_ms
    FROM iceberg.eval.eval_scores        s          -- lakehouse  (analytical scores)
    JOIN postgresql.public.eval_results  r          -- operational (live run context)
      ON s.result_id = r.id
    GROUP BY r.model, r.backend, s.metric
    ORDER BY n DESC
    LIMIT 15
''')
print(f"{fed.height} rows joined across iceberg + postgresql")
fed

15 rows joined across iceberg + postgresql


model,backend,metric,n,avg_score,avg_ms
str,str,str,i64,f64,f64
"""qwen3:30b-a3b""","""pgvector""","""faithfulness""",669,0.72,31397.0
"""qwen3:30b-a3b""","""pgvector""","""answer_relevancy""",669,0.763,31397.0
"""qwen3:30b-a3b""","""pgvector""","""context_relevancy""",669,0.654,31397.0
"""mistral-small3.2:24b""","""pgvector""","""faithfulness""",660,0.701,24610.0
"""mistral-small3.2:24b""","""pgvector""","""context_relevancy""",660,0.642,24610.0
…,…,…,…,…,…
"""gpt-oss:20b""","""pgvector""","""answer_relevancy""",657,0.785,15428.0
"""gpt-oss:20b""","""pgvector""","""context_relevancy""",655,0.671,15438.0
"""qwen3:14b""","""pgvector""","""faithfulness""",647,0.747,31531.0


Read that frame as one picture the mesh could not otherwise show in a single query:
the **average judge score per metric** (from the lakehouse) sits right next to the
**average latency** (from the operational DB) for each `model` × `backend`. Analytical
truth and operational truth, aligned on the same rows, no ETL. That is federation earning
its keep.

## Pushdown — Trino sends work *down* to the source

Federation would be slow if Trino dragged whole tables across the wire and filtered them
itself. It doesn't: the Trino Postgres connector **pushes predicates and column
projections down into Postgres**, so the remote database does the narrowing and only the
surviving, trimmed rows travel back.

`EXPLAIN` shows the plan. Look at the `ScanProject`/`TableScan` over
`postgresql:public.eval_results`: the `WHERE backend = 'pgvector'` filter appears as a
**`constraint on [backend]`** attached to the scan (predicate pushdown), and the scan lists
only **`columns=[model, latency_ms]`** — the two columns the query actually needs (column
pruning). Neither the filter nor the unused columns ever cross the wire.

In [9]:
plan = scalar('''
    EXPLAIN
    SELECT model, count(*) AS n, avg(latency_ms) AS avg_ms
    FROM postgresql.public.eval_results
    WHERE backend = 'pgvector'
    GROUP BY model
''')
# print the scan line(s) where the pushdown is visible, plus a little context
for line in plan.splitlines():
    if any(k in line for k in ("ScanProject", "TableScan", "constraint on", "columns=")):
        print(line.strip())

└─ ScanProject[table = postgresql:public.eval_results public.eval_results constraint on [backend] columns=[model:varchar:text, latency_ms:integer:int4]]


The `constraint on [backend]` is the filter executing **inside Postgres**, and the
short `columns=[...]` list is Trino asking Postgres for only those fields. Why it matters:
less data crosses the network, the remote source uses its own indexes, and the federated
join stays cheap even when one side is a large operational table. (The full `EXPLAIN` plan
is available above by printing `plan` in its entirety.)

## When to reach for Trino federation

**Reach for Trino when the value is in *combining* sources, ad hoc:**
- **Cross-engine joins with no pipeline** — exactly the centerpiece above: analytical
  lakehouse tables joined to a live operational database in one `SELECT`, nothing copied.
- **Explore-then-decide** — browse catalogs/schemas and query across them before you commit
  to building any ETL. If a join proves valuable, *then* consider materializing it.
- **BI over many stores** — one SQL surface (and one JDBC endpoint) for dashboards that
  need data spanning systems, without a warehouse loading step in front.

**Reach for a native client instead when you're on a single store's hot path** — a
latency-sensitive point lookup or write belongs on that store's own driver, not a
federated planner. That's the subject of **notebook `22`**, and it matters here for one
specific reason:

> **The Tier-2 stores are *not* Trino catalogs on this coordinator.** ClickHouse,
> Cassandra, MongoDB, CockroachDB, TimescaleDB and MySQL are **not** wired as Trino
> connectors — `SHOW CATALOGS` returns only `iceberg`, `postgresql`, and `system`. Those
> stores are reached by their **native clients** in notebook `22`, not through Trino. Don't
> assume a store is federated just because it's in the mesh; this Trino federates the
> lakehouse and the operational Postgres, and that is the set.

**And reach for the storage notebooks (`10` lakeFS, `11` Nessie/Iceberg) when the job is
*versioning*, not querying** — branching, committing, time-travelling the data itself.
Trino reads the current state of what those layers store; it doesn't version it.

| you want to… | use |
|--------------|-----|
| join data across heterogeneous engines, ad hoc, no ETL | **Trino** (this notebook) |
| a fast point read/write on one specific store | **native client** (notebook `22`) |
| version / branch / time-travel the data itself | **lakeFS / Nessie+Iceberg** (`10` / `11`) |

Trino is the mesh's answer to "I need to ask a question that spans the whole platform,
and I need the answer now" — one SQL statement, many engines, no bytes moved that don't
have to be.